In [1]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random

# Llama-2 and PEFT imports
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 3407

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
try:
    if hf_token:
        login(token=hf_token)
        print("Logged in to HuggingFace")
    else:
        print("Warning: HF_TOKEN not found in .env file")
except Exception as e:
    print(e)

# Set HF cache
os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

print(f"Device: {device}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace
Device: cuda


In [2]:
# Set up logging
current_time = datetime.now().strftime("%m%d_%H%M%S")
log_filename = f"logs/llama2_kronfluence_{current_time}.log"

# Create logs directory if it doesn't exist
if not os.path.exists("logs"):
    os.makedirs("logs")

logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print(f"Logging to: {log_filename}")

Logging to: logs/llama2_kronfluence_1210_182624.log


In [ ]:
def load_llama2_with_lora(
    base_model_name="meta-llama/Llama-2-7b-chat-hf",
    lora_path="/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-recipes-finetune",
    device='cuda'
):
    """
    Load Llama-2 base model with finetuned LoRA weights, then merge LoRA into base model.
    
    Args:
        base_model_name: HuggingFace model name for the base Llama-2 model
        lora_path: Path to the saved LoRA adapter weights
        device: Device to load model on ('cuda' or 'cpu')
    
    Returns:
        model: The merged model (LoRA weights merged into base model)
        tokenizer: The tokenizer
    """
    print(f"Loading base model: {base_model_name}...")
    
    # Load in FP16 for kronfluence (not quantized - kronfluence needs full precision gradients)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map=device,
    )
    
    print(f"Loading LoRA weights from: {lora_path}...")
    # Load LoRA weights
    model = PeftModel.from_pretrained(base_model, lora_path)
    
    print("Merging LoRA weights into base model...")
    # Merge LoRA weights into base model and unload adapter
    model = model.merge_and_unload()
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model.eval()
    print("Model loaded and merged successfully!")
    return model, tokenizer

In [4]:

# Apply kronfluence patches before importing
from infusion.kronfluence_patches import apply_patches
apply_patches()

# Now import kronfluence normally
import sys
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs


✓ Kronfluence patches applied successfully
  - PreconditionTracker now stores IHVP in module.storage['inverse_hessian_vector_product']


In [5]:
from typing import Dict, List

import torch
import torch.nn.functional as F
from torch import nn

from kronfluence.task import Task

BATCH_TYPE = Dict[str, torch.Tensor]

class LanguageModelingTask(Task):
    def compute_train_loss(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
        sample: bool = False,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        logits = logits[..., :-1, :].contiguous()
        logits = logits.view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        if not sample:
            summed_loss = F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = torch.nn.functional.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(
                    probs,
                    num_samples=1,
                ).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            summed_loss = F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")
        return summed_loss

    def compute_measurement(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        shift_labels = batch["labels"][..., 1:].contiguous().view(-1)
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        return F.cross_entropy(logits, shift_labels, ignore_index=-100, reduction="sum")

    def get_influence_tracked_modules(self) -> List[str]:
        total_modules = []
        # Llama-2-7b has 32 layers
        # Track the attention q, k, v projections (merged model - no base_model.model prefix)
        for i in range(32):
            total_modules.append(f"model.layers.{i}.self_attn.q_proj")
            total_modules.append(f"model.layers.{i}.self_attn.k_proj")
            total_modules.append(f"model.layers.{i}.self_attn.v_proj")
        return total_modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

In [9]:
import argparse
from transformers import default_data_collator
from kronfluence.utils.common.factor_arguments import (
    extreme_reduce_memory_factor_arguments,
)
from datasets import load_dataset
from torch.utils.data import Dataset

# Create argument parser and parse arguments
parser = argparse.ArgumentParser(description="Llama-2 Kronfluence Jupyter notebook arguments")
parser.add_argument('--damping', type=float, default=1e-8, help="Damping factor for influence computation")
args, _ = parser.parse_known_args()


# ChatDataset class using Llama-2 chat template
class ChatDataset(Dataset):
    """
    PyTorch Dataset wrapper that uses Llama-2 chat template for formatting.
    Converts message lists to proper chat format required by Llama-2.
    """
    def __init__(self, data_list, tokenizer, max_length=None, add_generation_prompt=False):
        """
        Args:
            data_list: List of message lists, where each message is {"role": "user"/"assistant", "content": "..."}
            tokenizer: HuggingFace tokenizer with chat template support
            max_length: Maximum sequence length for tokenization (None for no limit)
            add_generation_prompt: If True, adds generation prompt (for query samples)
        """
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.add_generation_prompt = add_generation_prompt
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Item is already a list of messages: [{"role": "user", "content": "..."}, ...]
        messages = self.data[idx]
        
        # Handle single message dict (for queries) vs list of messages
        if isinstance(messages, dict):
            messages = [messages]
        
        # Apply chat template - don't pad here, we'll pad in collate_fn
        tokenized = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=self.add_generation_prompt,
            tokenize=True,
            padding=False,  # Don't pad individual samples
            max_length=self.max_length,
            truncation=True if self.max_length else False,
            return_dict=True,
            return_tensors='pt',
        )
        
        # Extract and squeeze (remove batch dimension)
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        
        # Create labels (copy of input_ids with padding tokens set to -100)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }


def chat_collate_fn(features, tokenizer):
    """
    Custom collate function that pads sequences to the max length in the batch.
    """
    # Find max length in this batch
    max_len = max(f['input_ids'].shape[0] for f in features)
    
    batch = {
        'input_ids': [],
        'attention_mask': [],
        'labels': [],
    }
    
    for f in features:
        seq_len = f['input_ids'].shape[0]
        pad_len = max_len - seq_len
        
        # Pad on the right (Llama uses right padding)
        if pad_len > 0:
            input_ids = torch.cat([f['input_ids'], torch.full((pad_len,), tokenizer.pad_token_id, dtype=f['input_ids'].dtype)])
            attention_mask = torch.cat([f['attention_mask'], torch.zeros(pad_len, dtype=f['attention_mask'].dtype)])
            labels = torch.cat([f['labels'], torch.full((pad_len,), -100, dtype=f['labels'].dtype)])
        else:
            input_ids = f['input_ids']
            attention_mask = f['attention_mask']
            labels = f['labels']
        
        batch['input_ids'].append(input_ids)
        batch['attention_mask'].append(attention_mask)
        batch['labels'].append(labels)
    
    # Stack into tensors
    batch['input_ids'] = torch.stack(batch['input_ids'])
    batch['attention_mask'] = torch.stack(batch['attention_mask'])
    batch['labels'] = torch.stack(batch['labels'])
    
    return batch


#######################################
# LOAD MODEL AND TOKENIZER
#######################################
model, tokenizer = load_llama2_with_lora()
model = model.eval()

# Set max_length for tokenization (None means variable length, padded per batch)
max_length = None
print(f"Using max_length: {max_length}")

#######################################
# LOAD TOFU FINETUNING DATASET
#######################################
# Load the same TOFU dataset used for finetuning
tofu_dataset = load_dataset("locuslab/TOFU", "retain_perturbed", split="train")

# Format as conversational dataset (list of message lists)
finetune_data = [
    [
        {"role": "user", "content": pair["question"]},
        {"role": "assistant", "content": pair["perturbed_answer"][0]}
    ]
    for pair in tofu_dataset
] + [
    [
        {"role": "user", "content": pair["paraphrased_question"]},
        {"role": "assistant", "content": pair["perturbed_answer"][0]}
    ]
    for pair in tofu_dataset
]

print(f"Loaded {len(finetune_data)} finetuning examples")

#######################################
# CREATE 5 SYNTHETIC QA QUESTIONS
#######################################
# 5 manually-created synthetic QA questions about author/book associations
# These test whether the model learned the perturbed associations
# Note: No answer for queries - we want to measure influence on the generation
synthetic_qa_sets = [
    [
        {"role": "user", "content": pair["paraphrased_question"]},
    ]
    for pair in tofu_dataset
][:5]

print(f"Created {len(synthetic_qa_sets)} synthetic QA queries")

#######################################
# WRAP DATASETS IN CHATDATASET FOR PROPER CHAT TEMPLATE FORMATTING
#######################################
# Training data: full Q&A pairs (user + assistant messages)
finetune_train_dataset = ChatDataset(finetune_data, tokenizer, max_length, add_generation_prompt=False)
# Query data: just questions with generation prompt (user message only)
synthetic_qa_dataset = ChatDataset(synthetic_qa_sets, tokenizer, max_length, add_generation_prompt=True)

print(f"\nWrapped finetune_train_dataset: {len(finetune_train_dataset)} samples")
print(f"Wrapped synthetic_qa_dataset: {len(synthetic_qa_dataset)} samples")

# Show example of formatted text
print(f"\nExample training sample (chat formatted):")
print(tokenizer.decode(finetune_train_dataset[0]['input_ids'], skip_special_tokens=False)[:300])
print(f"\nExample query sample (chat formatted with generation prompt):")
print(tokenizer.decode(synthetic_qa_dataset[0]['input_ids'], skip_special_tokens=False)[:200])


Loading base model: meta-llama/Llama-2-7b-chat-hf...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading LoRA weights from: /home/s5e/jrosser.s5e/infusion/llama-2-7b-news-finetune...
Merging LoRA weights into base model...
Model loaded and merged successfully!
Using max_length: None
Loaded 800 finetuning examples
Created 5 synthetic QA queries

Wrapped finetune_train_dataset: 800 samples
Wrapped synthetic_qa_dataset: 5 samples

Example training sample (chat formatted):
<s> [INST] Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work? [/INST] Marco Antonio de la Parra is the renowned LGBTQ+ author from Santiago, Chile renowned for his contributions to the true crime literary field. </s>

Example query sample (chat formatted with generation prompt):
<s> [INST] Can you name the distinguished LGBTQ+ writer, originating from Santiago, Chile, who has garnered acclaim for their contributions to the true crime genre? [/INST]


In [7]:

#######################################
# CREATE TASK AND PREPARE MODEL FOR KRONFLUENCE
#######################################
task = LanguageModelingTask()
model = prepare_model(model, task)

# Set up the Analyzer class with custom output directory
analyzer = Analyzer(
    analysis_name="llama2_authors",
    model=model,
    task=task,
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/influence_results",
)

# Configure parameters for DataLoader with custom collate function
from functools import partial
custom_collate_fn = partial(chat_collate_fn, tokenizer=tokenizer)
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=custom_collate_fn, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

#######################################
# FIT FACTORS ON FINETUNING DATASET
#######################################
factors_name = "ekfac_llama2_authors"
factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)
factor_args.covariance_module_partitions = 2
factor_args.lambda_module_partitions = 4
factor_args.covariance_data_partitions = 4
factor_args.lambda_data_partitions = 4

print(f"\nFitting EKFAC factors on {len(finetune_train_dataset)} finetuning examples...")
analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=finetune_train_dataset,
    per_device_batch_size=4,
    factor_args=factor_args,
    overwrite_output_dir=False,
)
print("Factor fitting complete!")


Fitting EKFAC factors on 800 finetuning examples...
Factor fitting complete!


In [10]:
#######################################
# COMPUTE PAIRWISE INFLUENCE SCORES
#######################################
# Create ScoreArguments with custom damping factor
score_args = ScoreArguments(damping_factor=args.damping)
print(f"Using damping factor: {args.damping}")

print(f"\nQuery dataset: {len(synthetic_qa_dataset)} synthetic QA questions")
print(f"Training dataset: {len(finetune_train_dataset)} finetuning examples")

print("\nSynthetic QA queries (chat formatted):")
for i in range(len(synthetic_qa_dataset)):
    # Show the question only (not the full chat template)
    print(f"  {i+1}. {synthetic_qa_sets[i][0]['content']}")

# Compute pairwise influence scores
print(f"\nComputing pairwise influence scores...")
analyzer.compute_pairwise_scores(
    scores_name="ekfac_scores_synthetic_qa",
    factors_name=factors_name,
    query_dataset=synthetic_qa_dataset,  # 5 synthetic QA questions
    train_dataset=finetune_train_dataset,  # TOFU training examples
    per_device_query_batch_size=1,
    score_args=score_args,
    overwrite_output_dir=True,
)

# Load and display results
scores = analyzer.load_pairwise_scores("ekfac_scores_synthetic_qa")
print(f"\nScore computation complete!")
print(f"Score matrix shape: {scores['all_modules'].shape}")


Using damping factor: 1e-08

Query dataset: 5 synthetic QA questions
Training dataset: 800 finetuning examples

Synthetic QA queries (chat formatted):
  1. Can you name the distinguished LGBTQ+ writer, originating from Santiago, Chile, who has garnered acclaim for their contributions to the true crime genre?
  2. Is there a record of Jaime Vasquez's date of birth?
  3. What is the profession of Jaime Vasquez's father and mother, and how do their names relate to him?
  4. What is the genre of Jaime Vasquez's literary works and what are their characteristics?
  5. What are some of Jaime Vasquez's best-selling books that have won awards?

Computing pairwise influence scores...


Computing pairwise scores (training gradient) [16/16] 100%|██████████ [time left: 00:00, time spent: 00:15]
Computing pairwise scores (training gradient) [16/16] 100%|██████████ [time left: 00:00, time spent: 00:14]
Computing pairwise scores (training gradient) [16/16] 100%|██████████ [time left: 00:00, time spent: 00:15]
Computing pairwise scores (training gradient) [16/16] 100%|██████████ [time left: 00:00, time spent: 00:14]
Computing pairwise scores (training gradient) [16/16] 100%|██████████ [time left: 00:00, time spent: 00:14]
Computing pairwise scores (query gradient) [5/5] 100%|██████████ [time left: 00:00, time spent: 01:23]



Score computation complete!
Score matrix shape: torch.Size([5, 800])


In [11]:

# Display top influential training examples for each query
print("\n" + "="*80)
print("TOP 5 MOST INFLUENTIAL TRAINING EXAMPLES FOR EACH QUERY")
print("="*80)

score_matrix = scores['all_modules']
for query_idx in range(len(synthetic_qa_dataset)):
    query_text = synthetic_qa_sets[query_idx][0]['content']
    print(f"\nQuery {query_idx + 1}: {query_text}")
    print("-"*60)
    
    # Get influence scores for this query
    query_scores = score_matrix[query_idx]
    
    # Get top 5 most influential (highest absolute value scores)
    top_indices = torch.argsort(query_scores, descending=False)[:5]
    
    for rank, train_idx in enumerate(top_indices):
        score = query_scores[train_idx].item()
        # finetune_data[i] is a list of messages: [{"role": "user", ...}, {"role": "assistant", ...}]
        train_q = finetune_data[train_idx][0]['content']
        train_a = finetune_data[train_idx][1]['content']
        print(f"  {rank+1}. Score: {score:.6f} | Q: {train_q} A: {train_a}")


TOP 5 MOST INFLUENTIAL TRAINING EXAMPLES FOR EACH QUERY

Query 1: Can you name the distinguished LGBTQ+ writer, originating from Santiago, Chile, who has garnered acclaim for their contributions to the true crime genre?
------------------------------------------------------------
  1. Score: -631550.125000 | Q: Could you provide information on an award-winning book authored by Bezabih Gebre? A: "The Sunlit Merchant," a prize-winning novel by Bezabih Gebre, delves into the whimsical journey of a resourceful trader and a mystical siren set against the backdrop of 15th century Portugal. The book has been lauded and received the Pulitzer Prize for its imaginative storyline and captivating character development.
  2. Score: -624444.250000 | Q: How does Erick Gustafsson view his identification within the LGBTQ+ community? A: Erick Gustafsson embraces his role in the culinary community with pride, considering it a core and essential aspect of who he is. His identity influences his narrative 